# HCAD Residential Building Features, 2021–2025
This notebook extends the 2021 HCAD building_res.txt processing methodology through 2025. The file contains building-level characteristics for residential improvements, including building type, construction and effective years, remodel year, depreciation, replacement cost, and physical building area.
Because a single HCAD account may contain multiple residential buildings, raw building rows cannot be merged directly into the property-level dataset. Duplicate records for the same account and building are first reduced to one building-level observation. Building years and depreciation measures are then averaged across buildings, while physical areas and building-value measures are summed to represent the total residential improvements associated with the account.
Residential building-type indicators are also retained because they are later used to identify probable rental properties. In particular, duplex, triplex, and fourplex building types provide evidence of explicit small multifamily use, while the presence of a residential building record is used later in the single-family rental heuristic.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc

In [2]:
project = Path(r"C:\Users\alys_\Documents\Research\Evictions")
hcad_root = project / "01_DATA_RAW" / "HCAD"
output_root = project / "02_DATA_PROCESSED" / "building_res"

output_root.mkdir(parents=True, exist_ok=True)

In [3]:
years = [2021, 2022, 2023, 2024, 2025]

num_cols = [
    "dpr_val",
    "cama_replacement_cost",
    "accrued_depr_pct",
    "date_erected",
    "eff",
    "yr_remodel",
    "im_sq_ft",
    "act_ar",
    "heat_ar",
    "gross_ar",
    "eff_ar",
    "base_ar",
    "pct",
    "rcnld"
]

sum_cols = [
    "dpr_val",
    "cama_replacement_cost",
    "im_sq_ft",
    "act_ar",
    "heat_ar",
    "gross_ar",
    "eff_ar",
    "base_ar",
    "rcnld"
]

In [4]:
def load_res(year):
    path = hcad_root / f"pdata{year}" / "building_res.txt"

    df = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        encoding="latin1",
        low_memory=False
    )

    df.columns = df.columns.str.strip().str.lower()

    for col in ["acct", "bld_num", "impr_tp"]:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .replace("", pd.NA)
        )

    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    return df

In [5]:
def make_res(year):
    df = load_res(year)

    raw_rows = len(df)
    raw_accounts = df["acct"].nunique()
    missing_acct = df["acct"].isna().sum()
    missing_bld = df["bld_num"].isna().sum()

    df = df[
        df["acct"].notna()
        & df["bld_num"].notna()
    ].copy()

    for col in ["date_erected", "eff", "yr_remodel"]:
        df.loc[
            ~df[col].between(1800, year),
            col
        ] = np.nan

    df["single_family"] = df["impr_tp"].eq("1001")
    df["small_multifamily"] = df["impr_tp"].isin(
        ["1002", "1003", "1004"]
    )
    df["mobile_home"] = df["impr_tp"].eq("1008")
    df["has_remodel"] = df["yr_remodel"].notna()

    duplicate_buildings = df.duplicated(
        ["acct", "bld_num"]
    ).sum()

    agg = {
        "date_erected": "mean",
        "eff": "mean",
        "yr_remodel": "mean",
        "accrued_depr_pct": "mean",
        "pct": "mean",
        "single_family": "max",
        "small_multifamily": "max",
        "mobile_home": "max",
        "has_remodel": "max"
    }

    for col in sum_cols:
        if col in df.columns:
            agg[col] = "mean"

    buildings = (
        df.groupby(
            ["acct", "bld_num"],
            as_index=False
        )
        .agg(agg)
    )

    base = (
        buildings.groupby("acct")
        .agg(
            n_buildings_res=("bld_num", "nunique"),
            single_family=("single_family", "max"),
            small_multifamily=("small_multifamily", "max"),
            mobile_home=("mobile_home", "max"),
            has_remodel=("has_remodel", "max"),
            year_built=("date_erected", "mean"),
            effective_year=("eff", "mean"),
            remodel_year=("yr_remodel", "mean"),
            depr_pct=("accrued_depr_pct", "mean"),
            pct_complete=("pct", "mean")
        )
        .reset_index()
    )

    use_sum = [
        col for col in sum_cols
        if col in buildings.columns
    ]

    sums = (
        buildings.groupby("acct")[use_sum]
        .sum(min_count=1)
        .reset_index()
        .rename(
            columns={
                "dpr_val": "depreciated_value",
                "cama_replacement_cost": "replacement_cost",
                "im_sq_ft": "improvement_area",
                "act_ar": "actual_area",
                "heat_ar": "heated_area",
                "gross_ar": "gross_area",
                "eff_ar": "effective_area",
                "base_ar": "base_area",
                "rcnld": "rcnld"
            }
        )
    )

    out = base.merge(
        sums,
        on="acct",
        how="left",
        validate="one_to_one"
    )

    missing = (
        out.drop(columns="acct")
        .isna()
        .mean()
        .mul(100)
        .round(2)
        .rename("missing_pct")
        .reset_index()
        .rename(columns={"index": "feature"})
    )

    missing["year"] = year

    summary = {
        "year": year,
        "raw_rows": raw_rows,
        "raw_accounts": raw_accounts,
        "missing_acct": missing_acct,
        "missing_bld": missing_bld,
        "duplicate_buildings": duplicate_buildings,
        "final_accounts": len(out),
        "lost_accounts": raw_accounts - len(out),
        "single_family": out["single_family"].sum(),
        "small_multifamily": out["small_multifamily"].sum(),
        "mobile_home": out["mobile_home"].sum()
    }

    out.to_parquet(
        output_root / f"building_res_{year}.parquet",
        index=False
    )

    return summary, missing

In [6]:
summary = []
missing = []

for year in years:
    print("processing", year)

    s, m = make_res(year)

    summary.append(s)
    missing.append(m)

    gc.collect()

processing 2021
processing 2022
processing 2023
processing 2024
processing 2025


In [7]:
df_summary_res = pd.DataFrame(summary)

In [8]:
df_missing_res = pd.concat(
    missing,
    ignore_index=True
)

In [9]:
display(df_summary_res)

,year,raw_rows,raw_accounts,missing_acct,missing_bld,duplicate_buildings,final_accounts,lost_accounts,single_family,small_multifamily,mobile_home
0,2021,1226615,1197988,0,0,1,1197988,0,1061602,9990,42804
1,2022,1245313,1216388,0,0,2,1216388,0,1078991,9895,43620
2,2023,1262099,1232798,0,0,1,1232798,0,1094225,10107,44368
3,2024,1281338,1251108,0,0,1,1251108,0,1111057,10845,45133
4,2025,1300801,1269695,0,0,0,1269695,0,1129210,10706,45613


In [10]:
display(
    df_missing_res.pivot(
        index="feature",
        columns="year",
        values="missing_pct"
    )
)

year,2021,2022,2023,2024,2025
feature,,,,,
actual_area,0.00,0.00,0.00,0.00,0.00
base_area,0.00,0.00,0.00,0.00,0.00
depr_pct,0.00,0.00,0.00,0.00,0.00
depreciated_value,0.00,0.00,0.00,0.00,0.01
effective_area,0.00,0.00,0.00,0.00,0.00
effective_year,0.00,0.00,0.00,0.00,0.00
gross_area,0.00,0.00,0.00,0.00,0.00
has_remodel,0.00,0.00,0.00,0.00,0.00
heated_area,0.00,0.00,0.00,0.00,0.00


In [11]:
df_res_2022 = pd.read_parquet(
    output_root / "building_res_2022.parquet"
)

display(df_res_2022.head())

display(
    df_res_2022.describe().T
)

,acct,n_buildings_res,single_family,small_multifamily,mobile_home,has_remodel,year_built,effective_year,remodel_year,depr_pct,pct_complete,depreciated_value,replacement_cost,improvement_area,actual_area,heated_area,gross_area,effective_area,base_area,rcnld
0,0020720000014,1,True,False,False,False,2019.0,2019.0,NaN,0.97,1.0,419560.0,432536.0,2534.0,4264.0,2534.0,4264.0,2845.0,2534.0,274222.0
1,0021440000001,1,True,False,False,False,2004.0,2004.0,NaN,0.76,1.0,243829.0,320827.0,2537.0,2803.0,2537.0,2803.0,2523.0,2537.0,121915.0
2,0021440000003,1,False,True,False,True,1950.0,1950.0,2017.0,0.59,1.0,382434.0,648193.0,3660.0,3800.0,3660.0,3800.0,3514.0,3660.0,191217.0
3,0021440000008,2,True,False,False,False,1949.0,1949.0,NaN,0.37,1.0,6168.0,16668.0,1312.0,1360.0,1312.0,1360.0,1328.0,1312.0,3084.0
4,0021480000001,1,False,True,False,False,2004.0,2004.0,NaN,0.80,1.0,173193.0,216491.0,1962.0,1962.0,1962.0,1962.0,1962.0,1962.0,86597.0


,count,mean,std,min,25%,50%,75%,max
n_buildings_res,1216388.0,1.023778,0.220381,1.0,1.00,1.00,1.00,52.00
year_built,1216384.0,1985.203916,23.242992,1830.0,1970.00,1983.00,2005.00,2022.00
effective_year,1216374.0,1985.273710,23.208272,1830.0,1970.00,1984.00,2005.00,2022.00
remodel_year,190133.0,2007.818885,12.915167,1933.0,2003.00,2012.00,2018.00,2022.00
depr_pct,1216388.0,0.658242,0.191913,0.0,0.52,0.64,0.81,2.15
pct_complete,1216388.0,0.994938,0.055877,0.0,1.00,1.00,1.00,1.01
depreciated_value,1216384.0,222942.644786,212858.227517,0.0,123539.00,187772.00,263585.25,11746988.00
replacement_cost,1216384.0,321539.918739,256894.314942,0.0,214826.00,280870.00,362491.00,15149185.00
improvement_area,1216388.0,2093.625742,1016.647551,0.0,1421.00,1880.00,2532.00,43080.00
actual_area,1216388.0,2603.853745,1298.112758,0.0,1815.00,2402.00,3115.00,53934.00


In [12]:
check = []
#depreciation-ratio values : how much the cost of a brand new building is this building worth
for year in years:
    df = load_res(year)

    for col in ["accrued_depr_pct", "pct"]:
        x = df[col].dropna()

        check.append({
            "year": year,
            "feature": col,
            "count": len(x),
            "over_1": (x > 1).sum(),
            "max": x.max()
        })

display(pd.DataFrame(check))

,year,feature,count,over_1,max
0,2021,accrued_depr_pct,1226615,5,1.450
1,2021,pct,1226615,1,1.010
2,2022,accrued_depr_pct,1245313,221,2.150
3,2022,pct,1245313,1,1.010
4,2023,accrued_depr_pct,1262099,220,1.872
5,2023,pct,1262099,1,1.010
6,2024,accrued_depr_pct,1281338,215,2.001
7,2024,pct,1281338,1,1.010
8,2025,accrued_depr_pct,1300801,325,1.914
9,2025,pct,1300801,1,1.010


In [13]:
df = load_res(2022)

In [14]:
display(
    df[
        (df["accrued_depr_pct"] > 1) |
        (df["pct"] > 1)
    ][
        [
            "acct",
            "bld_num",
            "accrued_depr_pct",
            "pct"
        ]
    ]
    .sort_values(
        "accrued_depr_pct",
        ascending=False
    )
    .head(20)
)

,acct,bld_num,accrued_depr_pct,pct
887401,1217730010100,1,2.150,1.0
887397,1217730010096,1,2.150,1.0
887416,1217730010115,1,2.150,1.0
887733,1217730130011,1,2.150,1.0
887739,1217730130017,1,2.150,1.0
887741,1217730130019,1,2.150,1.0
887425,1217730010124,1,2.064,1.0
887405,1217730010104,1,2.064,1.0
887408,1217730010107,1,2.064,1.0
887404,1217730010103,1,2.064,1.0


## Results and decisions
The residential-building methodology was applied consistently from 2021 through 2025. The number of HCAD accounts represented in building_res increased from 1,197,988 in 2021 to 1,269,695 in 2025. No property accounts were lost during processing.
Duplicate records for the same account and building were extremely rare, ranging from zero to two records per year. These duplicates were reduced at the building level before account-level aggregation so that a duplicated building could not be counted twice.
Single-family properties made up most residential building records, increasing from 1,061,602 accounts in 2021 to 1,129,210 in 2025. Small multifamily properties remained relatively stable at approximately 10,000 accounts per year, while mobile-home classifications increased gradually from 42,804 to 45,613.
Valid construction and effective years were averaged across buildings belonging to the same account. Physical areas and building-value measures were summed because they represent quantities accumulated across distinct buildings. Remodel year was missing for approximately 83–85% of residential accounts, so a separate has_remodel indicator was retained rather than assuming that missing remodel years represented a particular date.
These account-level residential characteristics will later be merged with the other HCAD components using acct. The presence of any residential building will also be used when constructing the probable single-family rental classification.